# Notebook 3: Evaluation & Plots

**Settings:** GPU P100

**Add Inputs:** Notebook 1 output + Notebook 2 output (via Notebook Output tab)

In [ ]:
import os, copy, json, random
import numpy as np
import torch, torch.nn as nn
import torchvision.models as models, torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from scipy import stats

random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
for d in ['models','results/fedavg','results/fedprox','results/qpso','results/plots']:
    os.makedirs(f'/kaggle/working/{d}', exist_ok=True)

class BrainTumorDataset(Dataset):
    _TF = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        img = Image.fromarray((self.X[idx]*255).astype(np.uint8))
        return self._TF(img), torch.tensor(self.y[idx], dtype=torch.long)

def create_data_loaders(pdir="/kaggle/working/data/processed", tdir="/kaggle/working/data/test_set", bs=32):
    loaders = {}
    for i in range(1, 4):
        d = f"{pdir}/client{i}"
        for s in ['train','val','test']:
            ds = BrainTumorDataset(np.load(f"{d}/X_{s}.npy"), np.load(f"{d}/y_{s}.npy"))
            loaders.setdefault(f"client{i}", {})[s] = DataLoader(ds, bs, shuffle=(s=='train'), num_workers=2, pin_memory=True)
        loaders[f"client{i}"]["train_size"] = len(np.load(f"{d}/X_train.npy"))
    gds = BrainTumorDataset(np.load(f"{tdir}/X_test.npy"), np.load(f"{tdir}/y_test.npy"))
    loaders["global_test"] = DataLoader(gds, bs, shuffle=False, num_workers=2, pin_memory=True)
    return loaders

class BrainTumorResNet(nn.Module):
    def __init__(self, n=3):
        super().__init__()
        self.model = models.resnet18(weights=None)
        self.model.fc = nn.Linear(self.model.fc.in_features, n)
    def forward(self, x): return self.model(x)

CLASS_NAMES = ["Glioma", "Meningioma", "Pituitary"]

def plot_accuracy_comparison(dfs, names, save_path):
    n = len(dfs)
    fig, axes = plt.subplots(2, max(n,2), figsize=(8*max(n,2), 12))
    fig.suptitle("FL Aggregation Comparison", fontsize=16, fontweight="bold")
    colors = ['#1f77b4','#ff7f0e','#2ca02c']
    ax = axes[0,0]
    for df, name, c in zip(dfs, names, colors):
        ax.plot(df["round"], df["global_test_acc"], label=name, lw=2, color=c)
    ax.set(xlabel="Round", ylabel="Acc (%)", title="Global Test Accuracy", ylim=[0,100]); ax.legend(); ax.grid(alpha=0.3)
    ax = axes[0,1]
    for df, name, c in zip(dfs, names, colors):
        ax.plot(df["round"], df["global_test_loss"], label=name, lw=2, color=c)
    ax.set(xlabel="Round", ylabel="Loss", title="Global Test Loss"); ax.legend(); ax.grid(alpha=0.3)
    for idx, (df, name) in enumerate(zip(dfs, names)):
        ax = axes[1, idx]
        for i in range(1,4): ax.plot(df["round"], df[f"client{i}_val_acc"], label=f"Client {i}", lw=2)
        ax.set(xlabel="Round", ylabel="Val Acc (%)", title=f"{name}: Per-Client"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.show()
    print(f"✅ Saved -> {save_path}")

def generate_confusion_matrix(model_path, test_loader, method_name, device="cuda", save_path=None):
    model = BrainTumorResNet(3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for imgs, labs in test_loader:
            _, pred = model(imgs.to(device)).max(1)
            preds.extend(pred.cpu().numpy()); labels.extend(labs.numpy())
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix — {method_name}", fontsize=14, fontweight="bold")
    plt.ylabel("True"); plt.xlabel("Predicted"); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"\n{method_name} Report:")
    print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

def plot_roc_auc(model_paths, method_names, test_loader, device="cuda", save_path=None):
    from sklearn.preprocessing import label_binarize
    from sklearn.metrics import roc_curve, auc
    n = len(model_paths)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6))
    if n == 1: axes = [axes]
    colors_cls = ['#e74c3c', '#3498db', '#2ecc71']
    for ax, mp, mname in zip(axes, model_paths, method_names):
        model = BrainTumorResNet(3).to(device)
        model.load_state_dict(torch.load(mp, map_location=device, weights_only=True))
        model.eval(); all_probs, all_labels = [], []
        with torch.no_grad():
            for imgs, labs in test_loader:
                out = model(imgs.to(device))
                probs = torch.softmax(out, dim=1)
                all_probs.append(probs.cpu().numpy()); all_labels.extend(labs.numpy())
        y_prob = np.vstack(all_probs); y_true = np.array(all_labels)
        y_bin = label_binarize(y_true, classes=[0,1,2])
        for i, (cls, c) in enumerate(zip(CLASS_NAMES, colors_cls)):
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=c, lw=2, label=f"{cls} (AUC={roc_auc:.4f})")
        fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
        ax.plot(fpr_micro, tpr_micro, 'k--', lw=2, label=f"Micro-avg (AUC={auc(fpr_micro,tpr_micro):.4f})")
        ax.plot([0,1],[0,1],'gray',ls=':', lw=1)
        ax.set(xlabel="FPR", ylabel="TPR", title=f"ROC — {mname}")
        ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"✅ ROC-AUC saved -> {save_path}")

def plot_client_fairness(dfs, names, save_path):
    n = len(dfs)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5)); colors = ["#FF6B6B","#4ECDC4","#45B7D1"]
    if n == 1: axes = [axes]
    for ax, df, t in zip(axes, dfs, names):
        accs = [df[f"client{i}_val_acc"].iloc[-1] for i in range(1,4)]
        ax.bar(["C1","C2","C3"], accs, color=colors, edgecolor="black")
        ax.axhline(np.mean(accs), color="red", ls="--", lw=2, label="Mean")
        ax.set(ylabel="Final Val Acc (%)", ylim=[0,100])
        ax.set_title(f"{t} (σ={np.std(accs):.2f}%)", fontsize=13, fontweight="bold"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.show()

def convergence_metrics(df, target=80.0):
    reached = df[df["global_test_acc"]>=target]
    return {"final_acc": round(df["global_test_acc"].iloc[-1],2),
            "best_acc": round(df["global_test_acc"].max(),2),
            "round_to_target": int(reached["round"].min()) if len(reached) else None,
            "avg_round_time_s": round(df["round_time"].mean(),2),
            "total_time_min": round(df["round_time"].sum()/60,2)}

def statistical_analysis(df_fa, df_qp):
    a, b = df_fa["global_test_acc"].values, df_qp["global_test_acc"].values
    n = min(len(a),len(b)); a, b = a[:n], b[:n]
    t_stat, p_val = stats.ttest_rel(b, a); d = (b-a).mean()/((b-a).std(ddof=1)+1e-8)
    return {"t_stat": round(float(t_stat),4), "p_value": float(p_val),
            "cohens_d": round(float(d),4), "significant": bool(p_val<0.05)}

def build_comparison_df(*dfs_and_names):
    rows = []
    cols = ["Metric"] + [n for _, n in dfs_and_names]
    for _, name in dfs_and_names:
        pass  # just to get names
    metrics_list = []
    for df, name in dfs_and_names:
        m = convergence_metrics(df)
        m["client_std"] = round(float(np.std([df[f"client{i}_val_acc"].iloc[-1] for i in range(1,4)])),2)
        metrics_list.append(m)
    metric_keys = [("Final Acc (%)","final_acc"),("Best Acc (%)","best_acc"),
                   ("Rounds to 80%","round_to_target"),("Avg Round (s)","avg_round_time_s"),
                   ("Total Time (min)","total_time_min"),("Client Std Dev","client_std")]
    for label, key in metric_keys:
        row = [label] + [m[key] if m[key] is not None else "N/A" for m in metrics_list]
        rows.append(row)
    return pd.DataFrame(rows, columns=cols)

print("✅ All code loaded")

In [ ]:
import shutil

DATA_SRC_NB1 = '/kaggle/input/datasets/divyanshtejaedla/fl-dataset-brain-tumour-classification/data'
DATA_SRC_NB2 = '/kaggle/input/datasets/divyanshtejaedla/fl-qpso-training-results'
print("Starting to load data...")

# 1. Load Data from Notebook 1
if os.path.exists(DATA_SRC_NB1):
    shutil.copytree(DATA_SRC_NB1, '/kaggle/working/data', dirs_exist_ok=True)
    print(f"✅ Data copied from {DATA_SRC_NB1}")
else:
    print(f"⚠️ Could not find precise data path: {DATA_SRC_NB1}")

# 2. Load Models/Results from Notebook 2
if os.path.exists(DATA_SRC_NB2):
    for sub in ['models', 'results', 'logs']:
        sub_src = os.path.join(DATA_SRC_NB2, sub)
        if os.path.exists(sub_src):
            shutil.copytree(sub_src, f'/kaggle/working/{sub}', dirs_exist_ok=True)
            print(f"✅ Copied {sub} from {DATA_SRC_NB2}")
else:
    print(f"⚠️ Could not find precise models path: {DATA_SRC_NB2}")

# 3. Fallback: Search all inputs just in case
print("\nScanning /kaggle/input for any missing directories...")
for base_name in os.listdir('/kaggle/input'):
    if 'divyanshtejaedla' in base_name: continue # Already checked
    src = f'/kaggle/input/{base_name}'
    for sub in ['data','models','results','logs']:
        sub_src = os.path.join(src, sub)
        if os.path.exists(sub_src):
            shutil.copytree(sub_src, f'/kaggle/working/{sub}', dirs_exist_ok=True)
            print(f"  Copied fallback {base_name}/{sub}")

print("\nInitializing Data Loaders...")
data_loaders = create_data_loaders()
print("✅ Data loaded")

In [ ]:
df_fa = pd.read_csv('/kaggle/working/results/fedavg/metrics.csv')
df_fp = pd.read_csv('/kaggle/working/results/fedprox/metrics.csv')
df_qp = pd.read_csv('/kaggle/working/results/qpso/metrics.csv')
comp = build_comparison_df((df_fa,"FedAvg"),(df_fp,"FedProx"),(df_qp,"QPSO-FL"))
print(comp.to_string(index=False))
comp.to_csv('/kaggle/working/results/final_comparison.csv', index=False)

In [ ]:
plot_accuracy_comparison([df_fa, df_fp, df_qp], ['FedAvg','FedProx','QPSO-FL'], '/kaggle/working/results/plots/comprehensive_comparison.png')
generate_confusion_matrix('/kaggle/working/models/fedavg_best.pth', data_loaders['global_test'], 'FedAvg', device=device, save_path='/kaggle/working/results/plots/cm_fedavg.png')
generate_confusion_matrix('/kaggle/working/models/fedprox_best.pth', data_loaders['global_test'], 'FedProx', device=device, save_path='/kaggle/working/results/plots/cm_fedprox.png')
generate_confusion_matrix('/kaggle/working/models/qpso_best.pth', data_loaders['global_test'], 'QPSO-FL', device=device, save_path='/kaggle/working/results/plots/cm_qpso.png')
plot_client_fairness([df_fa, df_fp, df_qp], ['FedAvg','FedProx','QPSO-FL'], '/kaggle/working/results/plots/client_fairness.png')
plot_roc_auc(
    ['/kaggle/working/models/fedavg_best.pth', '/kaggle/working/models/fedprox_best.pth', '/kaggle/working/models/qpso_best.pth'],
    ['FedAvg', 'FedProx', 'QPSO-FL'], data_loaders['global_test'], device=device,
    save_path='/kaggle/working/results/plots/roc_auc.png')

In [ ]:
st_fa_qp = statistical_analysis(df_fa, df_qp)
st_fa_fp = statistical_analysis(df_fa, df_fp)
print("FedAvg vs QPSO:", json.dumps(st_fa_qp, indent=2))
print("FedAvg vs FedProx:", json.dumps(st_fa_fp, indent=2))
summary = {"fedavg": convergence_metrics(df_fa), "fedprox": convergence_metrics(df_fp),
           "qpso": convergence_metrics(df_qp),
           "stats_fedavg_vs_qpso": st_fa_qp, "stats_fedavg_vs_fedprox": st_fa_fp}
with open('/kaggle/working/results/executive_summary.json','w') as f: json.dump(summary, f, indent=2)
print("✅ Summary saved")
print("\nLaTeX Table:")
print(comp.to_latex(index=False, float_format="%.2f", caption="FedAvg vs FedProx vs QPSO-FL", label="tab:comparison"))
print("\n✅ Notebook 3 complete!")